# ReLU-Tune Training Run

Notebook for clone, install, train, evaluate, measure sparsity, and print final metrics.

### Downloading the Repo

In [ ]:
import os
from getpass import getpass 
# getpass keeps tokens out of notebook cell output.

REPO_URL = "https://github.com/BishmoyPaul/ReLU-Tune.git"
REPO_REF = "main"

!git clone $REPO_URL ReLU-Tune
%cd ReLU-Tune
!git fetch origin
!git checkout $REPO_REF

### Setting up Secrets

In [ ]:
## For Google Colab

from google.colab import userdata
HF_TOKEN = userdata.get("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN

## Optional wandb setup. Leave disabled unless you explicitly want it.
os.environ["WANDB_ENABLED"] = "false" # "false"
if os.environ.get("WANDB_ENABLED") == "true":
  WANDB_API_KEY = userdata.get("WANDB") # None
  os.environ["WANDB_API_KEY"] = WANDB_API_KEY


In [ ]:
# # For Other Cloud Services
# # Set the tokens in the environment
# HF_TOKEN = os.environ.get("HF_TOKEN")
# if not HF_TOKEN:
#     HF_TOKEN = getpass("HF_TOKEN: ").strip()

# os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN

# os.environ["WANDB_ENABLED"] = "false" # "true"
# if os.environ.get("WANDB_ENABLED") == "true":
#     # Optional wandb setup. Leave disabled unless you explicitly want it.
#     if not os.environ.get("WANDB_TOKEN"):
#         WANDB_API_KEY = getpass("WANDB_API_KEY: ").strip()
#     else:
#         WANDB_API_KEY = os.environ.get("WANDB_TOKEN")
    
#     os.environ["WANDB_API_KEY"] = WANDB_API_KEY


### Setting up the repo

In [ ]:
!pip uninstall -y torchao # for google colab
!pip install -r requirements.txt

## Config

In [ ]:
# Default debug preset for learning the workflow.
# Switch to configs/train_llama31_8b.yaml for the full 8B run.
TRAIN_CONFIG = "configs/train_llama32_1b_debug.yaml"
EVAL_CONFIG = "configs/evaluation.yaml"
SPARSITY_CONFIG = "configs/sparsity.yaml"
USE_TIMESTAMP = True

In [ ]:
from datetime import datetime
from pathlib import Path

import yaml

train_config_path = Path(TRAIN_CONFIG)
runtime_train_config_path = Path("configs/train_runtime.yaml")
train_config = yaml.safe_load(train_config_path.read_text(encoding="utf-8"))

if USE_TIMESTAMP:
    train_config["run_name"] = f"{train_config['run_name']}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

train_config["logging"]["wandb"]["enabled"] = os.environ.get("WANDB_ENABLED")
runtime_train_config_path.write_text(
    yaml.safe_dump(train_config, sort_keys=False),
    encoding="utf-8",
)
 
BASE_MODEL = train_config["model_id"]
RUN_DIR = str(Path(train_config.get("output_root", "./runs")) / train_config["run_name"])

print("Base model:", BASE_MODEL)
print("Run dir:", RUN_DIR)


## Training

In [ ]:
!python scripts/train_staged_lora.py --config {runtime_train_config_path} --run-dir {RUN_DIR}

In [ ]:
!python scripts/plot_run_metrics.py --run-dir {RUN_DIR}

In [ ]:
from IPython.display import Image, display
from pathlib import Path

plots_dir = Path(RUN_DIR) / "plots"
for name in ["train_loss.png", "val_loss.png", "loss_overlay.png", "lr.png"]:
    path = plots_dir / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))


## Evaluation

In [ ]:
!python scripts/evaluate.py --config {EVAL_CONFIG} --model-path {RUN_DIR}/final_merged --output-dir {RUN_DIR}/final_eval

In [ ]:
!python scripts/measure_sparsity.py --config {SPARSITY_CONFIG} --model-path {RUN_DIR}/final_merged --output-path {RUN_DIR}/final_eval/prefill_sparsity.json

## [Optional] Baseline Evaluation

In [ ]:
!python scripts/evaluate.py --config {EVAL_CONFIG} --base-model {BASE_MODEL} --output-dir {RUN_DIR}/baseline_eval

## [Optional] Baseline Sparsity

In [ ]:
!python scripts/measure_sparsity.py --config {SPARSITY_CONFIG} --base-model {BASE_MODEL} --output-path {RUN_DIR}/baseline_eval/prefill_sparsity.json

## Summary

In [ ]:
import json
from pathlib import Path

run_dir = Path(RUN_DIR)
def latest_json(directory):
    files = sorted(directory.glob("evaluation_*.json"))
    return files[-1] if files else None

def latest_metric(metric_name):
    latest = None
    for path in sorted(run_dir.glob("stage_*/metrics.json")):
        logs = json.loads(path.read_text(encoding="utf-8"))
        for entry in logs:
            if metric_name in entry:
                latest = entry
    return latest

def summarize_perplexity(payload):
    summary = {}
    for name, value in payload.items():
        if isinstance(value, dict) and "perplexity" in value:
            summary[name] = round(value["perplexity"], 4)
    return summary

def summarize_sparsity(path):
    if not path.exists():
        return None
    payload = json.loads(path.read_text(encoding="utf-8"))
    summary = payload.get("summary", {})
    average = summary.get("average_sparsity")
    if average is None:
        return None
    return {
        "average_sparsity_percent": round(average, 4),
        "measured_layers": summary.get("measured_layers"),
    }

last_train = latest_metric("loss")
last_eval = latest_metric("eval_loss")

print(f"Run dir: {run_dir}")

if last_train:
    print(f"Final train loss: {last_train['loss']:.4f} (step {last_train.get('step', 'n/a')})")
else:
    print("Final train loss: not found")

if last_eval:
    print(f"Final val loss: {last_eval['eval_loss']:.4f} (step {last_eval.get('step', 'n/a')})")
else:
    print("Final val loss: not found")

merged_eval_path = latest_json(run_dir / "final_eval")
if merged_eval_path:
    merged_eval = json.loads(merged_eval_path.read_text(encoding="utf-8"))
    print("\nMerged model outputs:")
    print(f"- evaluation file: {merged_eval_path.name}")
    benchmark_summary = merged_eval.get("benchmarks", {}).get("summary")
    if benchmark_summary:
        print(f"- benchmark summary: {benchmark_summary}")
    perplexity_summary = summarize_perplexity(merged_eval.get("perplexity", {}))
    if perplexity_summary:
        print(f"- perplexity: {perplexity_summary}")
else:
    print("\nMerged model outputs: not found")

merged_sparsity = summarize_sparsity(run_dir / "final_eval" / "prefill_sparsity.json")
if merged_sparsity:
    print(f"- sparsity: {merged_sparsity}")

baseline_eval_path = latest_json(run_dir / "baseline_eval")
baseline_sparsity = summarize_sparsity(run_dir / "baseline_eval" / "prefill_sparsity.json")
if baseline_eval_path or baseline_sparsity:
    print("\nOptional baseline outputs found in baseline_eval/.")
